In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
print(os.getcwd())
os.chdir("C://Users//Usuario//OneDrive - Global Green Growth Institute//Documentos//2025//Outputs//Output1//Stress Test//3.Data")

c:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python


In [3]:
datos = pd.read_csv(
    "sfc_322.csv",
    decimal=",",
    low_memory=False
)

list_of_columns=["Cartera de créditos","Créditos de vivienda"]

for i in list_of_columns:
    datos[i] = (
        datos[i]
            .str.replace(".", "", regex=False)   
            .str.replace(",", ".", regex=False)
            .astype(float)
    )
datos.head()

,Tipo de entidad,Código de entidad,Código del departamento,Código del municipio,Nombre del municipio,Fecha de Corte,Depósitos en cuenta corriente bancaria,Depósitos simples,Certificados de depósito a término,Depósitos de ahorro,...,Número de CDT,Número de cuentas centralizadas,Número de oficinas,Número de empleados contratados,Número de empleados subcontratados,Componente contracíclico provisión individual,Derechos de transferencia de cartera de créditos por operaciones de apoyos transitorios de liquidez,Préstamos a empleados,Deterioro (provisión) préstamos a empleados,Crédito de consumo de bajo monto
0,1,43,8,824,TOTORO,31/03/2006,"518.517.130,03",0,19.800.000,"322.946.893,15",...,0,0,0,18.131.663,7.542.349,5.621.906,1.244.002,0,0,0
1,1,1,15,245,EL BANCO,31/03/2006,1.871.890.046,0,949.826.261,"3.264.305.420,93",...,0,0,0,0,22.760.299,4.935.148,5.401.739,29.707.752,0,0
2,1,43,7,1,FLORENCIA,31/03/2006,"3.038.650.058,91",0,"422.373.421,63","4.198.312.255,1",...,0,0,0,73.089.887,16.986.009,56.537.436,99.104.495,0,0,0
3,1,43,10,419,LOS CORDOBAS,31/03/2006,"171.211.796,71",0,2.000.000,"84.009.894,94",...,0,0,0,15.036,25.666,17.092.734,152.063.902,0,0,0
4,1,34,22,1,SINCELEJO,31/03/2006,"888.670.618,84",0,"953.914.493,18","478.555.792,33",...,0,0,0,0,"138.124.035,85",0,"1.084.776,69",0,0,0


In [9]:
datos=datos[datos["Fecha de Corte"]=='30/09/2025']
print(ultimo_corte.shape)

(7613, 80)


In [15]:
cols_riesgo_totales = [
    'Categoría A riesgo normal',
    'Categoría B riesgo aceptable',
    'Categoría C riesgo apreciable',
    'Categoría D riesgo significativo',
    'Categoría E riesgo de Incobrabilidad',
]

def limpiar_numerico(series):
    if series.dtype == object:
        return (
            series.astype(str)
                  .str.replace(".", "", regex=False)
                  .str.replace(",", ".", regex=False)
                  .replace("nan", np.nan)
                  .astype(float)
        )
    return series.astype(float)

for col in cols_riesgo_totales:
    datos[col] = limpiar_numerico(datos[col])
datos[cols_riesgo_totales].dtypes

Categoría A riesgo normal               float64
Categoría B riesgo aceptable            float64
Categoría C riesgo apreciable           float64
Categoría D riesgo significativo        float64
Categoría E riesgo de Incobrabilidad    float64
dtype: object

In [16]:
ultimo_corte = datos[datos["Fecha de Corte"] == '30/09/2025'].copy()
print(f"Registros en último corte: {ultimo_corte.shape[0]}")

ultimo_corte[['Cartera de créditos'] + cols_riesgo_totales].describe()

Registros en último corte: 7613


,Cartera de créditos,Categoría A riesgo normal,Categoría B riesgo aceptable,Categoría C riesgo apreciable,Categoría D riesgo significativo,Categoría E riesgo de Incobrabilidad
count,7.613000e+03,7.613000e+03,7.613000e+03,7.613000e+03,7.613000e+03,7.613000e+03
mean,2.438502e+11,4.234189e+10,1.063291e+09,5.663827e+08,8.301511e+08,6.859894e+08
std,3.270986e+12,6.063671e+11,1.640555e+10,8.662471e+09,1.399809e+10,9.287890e+09
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.643256e+07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.116150e+09,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2.805855e+10,3.427738e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,1.811897e+14,3.249038e+13,1.049419e+12,5.281022e+11,8.868793e+11,4.544761e+11


In [ ]:
cols_agrupar = ['Código de entidad', 'Código del departamento',
                'Código del municipio', 'Nombre del municipio']
cols_valor   = ['Cartera de créditos'] + cols_riesgo_totales

cartera_banco_mpio = (
    ultimo_corte[cols_agrupar + cols_valor]
    .groupby(cols_agrupar, as_index=False)
    .sum(numeric_only=True)
)

cartera_banco_mpio = cartera_banco_mpio.rename(columns={
    'Categoría A riesgo normal'           : 'Cartera_A_normal',
    'Categoría B riesgo aceptable'        : 'Cartera_B_aceptable',
    'Categoría C riesgo apreciable'       : 'Cartera_C_apreciable',
    'Categoría D riesgo significativo'    : 'Cartera_D_significativo',
    'Categoría E riesgo de Incobrabilidad': 'Cartera_E_incobrabilidad',
})

print(f"Filas en tabla banco y municipio: {cartera_banco_mpio.shape[0]}")
cartera_banco_mpio.head(10)

Filas en tabla banco-municipio: 7566


,Código de entidad,Código del departamento,Código del municipio,Nombre del municipio,Cartera de créditos,Cartera_A_normal,Cartera_B_aceptable,Cartera_C_apreciable,Cartera_D_significativo,Cartera_E_incobrabilidad
0,1,1,1,MEDELLIN,1.228066e+13,8.923542e+11,1.030636e+10,2.655921e+09,3.256719e+09,4.623366e+09
1,1,1,2,ABEJORRAL,3.326064e+10,6.323147e+09,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,1,1,34,ANDES,3.663770e+10,8.106156e+09,0.000000e+00,0.000000e+00,0.000000e+00,1.708359e+07
3,1,1,42,SANTAFE DE ANTIOQUIA,9.286054e+09,1.546860e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,1,1,45,APARTADO,2.080046e+11,2.881068e+10,1.630422e+09,1.480100e+08,9.590094e+08,1.367880e+09
5,1,1,79,BARBOSA,1.094423e+10,1.142871e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
6,1,1,88,BELLO,1.846747e+11,5.051482e+10,5.236932e+08,2.115345e+08,3.261568e+07,2.719416e+08
7,1,1,101,BOLIVAR,4.229047e+09,9.256430e+07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
8,1,1,107,BRICE#O,5.018397e+09,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
9,1,1,129,CALDAS,3.475115e+10,1.312317e+10,3.764881e+08,3.653186e+07,0.000000e+00,1.076626e+08


In [18]:
cartera_banco_mpio['suma_categorias'] = (
    cartera_banco_mpio['Cartera_A_normal'] +
    cartera_banco_mpio['Cartera_B_aceptable'] +
    cartera_banco_mpio['Cartera_C_apreciable'] +
    cartera_banco_mpio['Cartera_D_significativo'] +
    cartera_banco_mpio['Cartera_E_incobrabilidad']
)

cartera_banco_mpio['diferencia'] = (
    cartera_banco_mpio['Cartera de créditos'] - cartera_banco_mpio['suma_categorias']
)

print("Diferencia total (Cartera - suma A+B+C+D+E):")
print(f"  Suma total cartera      : {cartera_banco_mpio['Cartera de créditos'].sum():,.0f}")
print(f"  Suma total A+B+C+D+E    : {cartera_banco_mpio['suma_categorias'].sum():,.0f}")
print(f"  Diferencia              : {cartera_banco_mpio['diferencia'].sum():,.0f}")
print(f"\n% filas con diferencia > 0: {(cartera_banco_mpio['diferencia'].abs() > 1).mean()*100:.1f}%")

Diferencia total (Cartera - suma A+B+C+D+E):
  Suma total cartera      : 1,856,431,544,994,300
  Suma total A+B+C+D+E    : 346,297,860,668,069
  Diferencia              : 1,510,133,684,326,230

% filas con diferencia > 0: 95.2%


In [19]:
cartera_por_banco = (
    cartera_banco_mpio
    .groupby('Código de entidad', as_index=False)[[
        'Cartera de créditos',
        'Cartera_A_normal', 'Cartera_B_aceptable',
        'Cartera_C_apreciable', 'Cartera_D_significativo',
        'Cartera_E_incobrabilidad', 'suma_categorias', 'diferencia'
    ]]
    .sum(numeric_only=True)
    .sort_values('Cartera de créditos', ascending=False)
)

for cat in ['A_normal', 'B_aceptable', 'C_apreciable', 'D_significativo', 'E_incobrabilidad']:
    cartera_por_banco[f'Pct_{cat}'] = (
        cartera_por_banco[f'Cartera_{cat}'] / cartera_por_banco['Cartera de créditos'] * 100
    ).round(2)

print(f"Número de entidades: {cartera_por_banco.shape[0]}")
cartera_por_banco.head(10)

Número de entidades: 50


,Código de entidad,Cartera de créditos,Cartera_A_normal,Cartera_B_aceptable,Cartera_C_apreciable,Cartera_D_significativo,Cartera_E_incobrabilidad,suma_categorias,diferencia,Pct_A_normal,Pct_B_aceptable,Pct_C_apreciable,Pct_D_significativo,Pct_E_incobrabilidad
6,7,4.961454e+14,7.143889e+13,1.299335e+12,9.890555e+11,1.507332e+12,1.055363e+12,7.628997e+13,4.198554e+14,14.40,0.26,0.20,0.30,0.21
17,39,2.526818e+14,8.453093e+13,2.784782e+12,1.403623e+12,2.356837e+12,1.198855e+12,9.227503e+13,1.604068e+14,33.45,1.10,0.56,0.93,0.47
0,1,2.178778e+14,3.046700e+13,6.417149e+11,3.364409e+11,2.386531e+11,4.424541e+11,3.212626e+13,1.857516e+14,13.98,0.29,0.15,0.11,0.20
11,13,1.928153e+14,3.634540e+13,1.094671e+12,3.733400e+11,3.288207e+11,7.203146e+11,3.886255e+13,1.539527e+14,18.85,0.57,0.19,0.17,0.37
12,23,1.246450e+14,8.708138e+12,1.191596e+11,3.949138e+10,3.007430e+11,8.926054e+10,9.256793e+12,1.153882e+14,6.99,0.10,0.03,0.24,0.07
1,2,9.191124e+13,1.914556e+12,3.004839e+10,2.676743e+10,1.165389e+10,4.333003e+10,2.026356e+12,8.988488e+13,2.08,0.03,0.03,0.01,0.05
18,42,6.692461e+13,1.398498e+13,3.308338e+11,2.136226e+11,2.610191e+11,3.424683e+11,1.513293e+13,5.179169e+13,20.90,0.49,0.32,0.39,0.51
19,43,6.361067e+13,1.484460e+12,2.277983e+10,6.640493e+10,7.360526e+09,1.058310e+10,1.591589e+12,6.201908e+13,2.33,0.04,0.10,0.01,0.02
5,6,4.406721e+13,7.047470e+12,1.339967e+11,8.045018e+10,4.227813e+10,1.208492e+11,7.425044e+12,3.664217e+13,15.99,0.30,0.18,0.10,0.27
14,30,4.270046e+13,2.048743e+13,3.593400e+11,1.602189e+11,8.987839e+11,4.104907e+11,2.231626e+13,2.038420e+13,47.98,0.84,0.38,2.10,0.96


In [2]:
4.198554e+14

419855400000000.0

In [20]:
cartera_banco_mpio.to_csv(
    "cartera_banco_municipio_322.csv",
    index=False,
    decimal=","
)

cartera_por_banco.to_csv(
    "cartera_por_banco_322.csv",
    index=False,
    decimal=","
)

print("Archivos exportados:")
print("  - cartera_banco_municipio_322.csv  (nivel banco-municipio)")
print("  - cartera_por_banco_322.csv        (nivel banco, agregado nacional)")

Archivos exportados:
  - cartera_banco_municipio_322.csv  (nivel banco-municipio)
  - cartera_por_banco_322.csv        (nivel banco, agregado nacional)
